# Notebook 1 — Clustering the MD Trajectory

Assign every trajectory frame to a discrete microstate (cluster).  
The cluster-ID sequence is the input for committor counting (Notebook 2).

Two methods are provided:
1. **TICA + K-Means** (recommended) — project onto the slowest collective motion first, then cluster.
2. **Direct K-Means** on backbone coordinates.

**Output:** `data/<system>/cluster_<N>.csv` with columns `Time_ps, Cluster_ID`.

In [ ]:
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyemma
import pyemma.coordinates as coor
from deeptime.clustering import KMeans
from scipy.spatial.distance import cdist

In [ ]:
# --- Select system ---
with open("../config/chignolin.yaml") as f:   # change to ww_domain.yaml for WW domain
    cfg = yaml.safe_load(f)

PDB   = cfg["topology_pdb"]
FILES = cfg["trajectory_xtc"]
SYSTEM = cfg["system"]
print(f"System: {SYSTEM}")
print(f"PDB:    {PDB}")
print(f"Traj:   {FILES}")

## Method 1 — TICA + K-Means (recommended)

TICA (Time-lagged Independent Component Analysis) projects the trajectory  
onto the slowest-relaxing collective coordinates. K-Means on the TICA  
output gives more kinetically meaningful clusters than clustering raw  
coordinates.

**VAMP2 score** is used to select the best lag time.

In [ ]:
# Load raw coordinates (no featurization — use Cartesian coordinates)
output = coor.load(FILES, top=PDB, stride=1)
print(f"Loaded {output.shape[0]:,} frames")

In [ ]:
def score_cv(data, dim, lag, n_splits=5, val_frac=0.5):
    """Cross-validated VAMP2 score."""
    with pyemma.util.contexts.settings(show_progress_bars=False):
        nval = max(1, int(len(data) * val_frac))
        scores = []
        for _ in range(n_splits):
            ival = np.random.choice(len(data), size=nval, replace=False)
            vamp = coor.vamp([d for i, d in enumerate(data) if i not in ival],
                             lag=lag, dim=dim)
            scores.append(vamp.score([d for i, d in enumerate(data) if i in ival]))
    return np.array(scores)

# Evaluate lag times 3, 4, 5
for lag in [3, 4, 5]:
    s = score_cv(output, dim=1, lag=lag)
    print(f"lag={lag}  VAMP2={s.mean():.3f} ± {s.std():.3f}")

In [ ]:
# Set lag time based on VAMP2 results above (typically lag=3 for Chignolin)
TICA_LAG = 3
TICA_DIM = 1

tica = coor.tica(output, lag=TICA_LAG, dim=TICA_DIM)
tica_output = tica.get_output()
tica_data = np.concatenate(tica_output)
print(f"TICA output shape: {tica_data.shape}")

In [ ]:
# Elbow method to choose K
K_range = range(100, 1001, 100)
distortions = []
for k in K_range:
    model = KMeans(n_clusters=k)
    clustered = model.fit(tica_data)
    d = sum(np.min(cdist(tica_data, clustered.initial_centers, 'euclidean'), axis=1)) / len(tica_data)
    distortions.append(d)
    print(f"k={k}  distortion={d:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(list(K_range), distortions, 'bx-')
plt.xlabel('K (number of clusters)')
plt.ylabel('Mean distortion')
plt.title('Elbow method')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Run final clustering with chosen K
N_CLUSTERS = 400   # adjust based on elbow plot

model = KMeans(n_clusters=N_CLUSTERS)
clustered = model.fit(tica_data)
labels = clustered.transform(tica_data)

# Write cluster CSV
import os
out_dir = f"../data/{SYSTEM}"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/cluster_{N_CLUSTERS}.csv"

df_clusters = pd.DataFrame({"Time_ps": np.arange(len(labels)) * 200,
                             "Cluster_ID": labels})
df_clusters.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

## Method 2 — Direct K-Means on minimum-RMSD features

This alternative uses RMSD from the reference structure as the feature space,  
which is simpler but less kinetically motivated than TICA.

In [ ]:
feat = coor.featurizer(PDB)
feat.add_minrmsd_to_ref(PDB)
reader = pyemma.coordinates.source(FILES, features=feat)
rmsd_data = reader.get_output()[0]

N_CLUSTERS_RMSD = 40  # smaller cluster count for quick committor counting
model_rmsd = KMeans(n_clusters=N_CLUSTERS_RMSD)
clustered_rmsd = model_rmsd.fit(rmsd_data)
labels_rmsd = clustered_rmsd.transform(rmsd_data)

out_path_rmsd = f"{out_dir}/cluster_{N_CLUSTERS_RMSD}.csv"
pd.DataFrame({"Time_ps": np.arange(len(labels_rmsd)) * 200,
              "Cluster_ID": labels_rmsd}).to_csv(out_path_rmsd, index=False)
print(f"Saved: {out_path_rmsd}")